In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Set seed for reproducibility
np.random.seed(42)

# Using Path().resolve() for Notebook consistency (mimics __file__ for interactive environments)
# Traverse up two levels from training/mcmc to project root



PROJECT_ROOT = Path().resolve().parent.parent

DATA_DIR = PROJECT_ROOT / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

MOCK_CSV_PATH = DATA_DIR / "mock_drill_logs.csv"

In [2]:
n_boreholes = 25
data = {
    "borehole_id": [f"BH-{i+1:03d}" for i in range(n_boreholes)],
    "depth_m": np.random.uniform(10, 150, n_boreholes).round(1),
    "rmr_score": np.clip(np.random.normal(52, 12, n_boreholes), 0, 100).round(1),
    "rock_class": np.random.choice(["II", "III", "IV", "V"], n_boreholes, p=[0.1, 0.4, 0.35, 0.15]),
    "ucs_mpa": np.clip(np.random.lognormal(3.8, 0.4, n_boreholes), 5, 200).round(1),
    "rqd_pct": np.clip(np.random.beta(5, 3, n_boreholes) * 100, 0, 100).round(1),
}

# Inject some NaN values (missing data — reality!)
data["ucs_mpa"][3] = np.nan
data["ucs_mpa"][17] = np.nan
data["rqd_pct"][8] = np.nan

df = pd.DataFrame(data)

# Save to the defined path
df.to_csv(MOCK_CSV_PATH, index=False)
#print(f"Borehole data orchestrated to: {MOCK_CSV_PATH}")

df.head()

,borehole_id,depth_m,rmr_score,rock_class,ucs_mpa,rqd_pct
0,BH-001,62.4,44.4,III,36.0,51.6
1,BH-002,143.1,59.2,III,43.4,60.4
2,BH-003,112.5,82.7,IV,66.8,63.1
3,BH-004,93.8,56.7,III,NaN,52.2
4,BH-005,31.8,53.5,IV,34.5,56.1


In [ ]:
class DrillLogPipeline: # lets keep it super specific to our csv
    """Ingests, validates, and formats drill-log data for PyMC models."""

    def __init__(self, filepath: Path ) -> None:
        self.filepath = filepath
        self.raw_data = None
        self.clean_data = None


    def load(self) -> pd.DataFrame:
        """load the raw csv"""
        self.raw_data = pd.read_csv(self.filepath)
        print(f"Loaded {len(self.raw_data)} rows from {self.filepath}")
        return self.raw_data

    def check_null(self):
        """High-signal Forensic Audit: only reports on Data Voids (NaNs)."""
        null_counts = self.raw_data.isnull().sum()
        # Filter for only parameters containing voids
        self.data_voids = null_counts[null_counts > 0]

        if self.data_voids.empty:
            print("Forensic Audit: No data voids detected.")
        else:
            print("Forensic Alert: Data voids found in high-priority parameters:")
            for param, count in self.data_voids.items():
                print(f"  - {param}: {count} null values")
        return self.data_voids

    def check_range(self, column_targets: list | dict, min_val: float = 0, max_val: float = 100, clipto_bounds: bool = False):
        """
        Checks columns against physical ranges.
        Provide a list to use default min/max bounds, or a dict mapping targets to (min, max) tuples.
        """
        results = {}

        # Standardize input to dictionary mapping for uniform processing
        if isinstance(column_targets, list):
            column_targets = {col: (min_val, max_val) for col in column_targets}

        for target, (target_min, target_max) in column_targets.items():
            # Map index to name if integer provided
            col_name = self.raw_data.columns[target] if isinstance(target, int) else target

            # Vectorized Range Audit
            out_of_range = np.sum((self.raw_data[col_name] < target_min) | (self.raw_data[col_name] > target_max)).item()
            results[col_name] = out_of_range

            if out_of_range > 0:
                print(f"Forensic Alert: [{col_name}] has {out_of_range} values outside [{target_min}, {target_max}]")

            if not clipto_bounds:
                # Enforce physical bounds directly onto the dataframe
                self.raw_data[col_name] = self.raw_data[col_name].clip(lower=target_min, upper=target_max)
                if out_of_range > 0:
                  x  print(f"  -> Action Taken: Clipped out-of-bounds values for [{col_name}] to [{target_min}, {target_max}].")

        return results


    def clean(self, drop_na: bool = False) -> pd.DataFrame:
        """Clean the data for modeling."""
        df = self.raw_data.copy()

        if drop_na:
            # Dynamically identify columns containing Data Voids (NaNs)
            na_cols = df.columns[df.isnull().any()].tolist()
            if na_cols:
                df = df.dropna(subset=na_cols)
                print(f"Dropped rows with missing values in {na_cols}. Remaining: {len(df)}")

        self.clean_data = df
        return df

In [4]:
pipeline = DrillLogPipeline(MOCK_CSV_PATH)

pipeline.load()
pipeline.check_null()
pipeline.check_range([2, 4,5])


Loaded 25 rows from F:\S1CL\M7_risk_rock\data\mock_drill_logs.csv
Forensic Alert: Data voids found in high-priority parameters:
  - ucs_mpa: 2 null values
  - rqd_pct: 1 null values


{'rmr_score': 0, 'ucs_mpa': 0, 'rqd_pct': 0}